In [1]:
# Cài đặt thư viện kết nối Kafka
!pip install kafka-python pandas

import pandas as pd
import json
import time
from kafka import KafkaProducer

# 1. Khởi tạo Kafka Producer (kết nối với container Kafka qua cổng 9092)
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# 2. Định nghĩa đường dẫn file CSV
# Dựa vào cấu hình Docker lúc nãy, thư mục data_source của bạn đã được mount vào /home/iceberg/data
csv_file_path = '/home/iceberg/data/olist_customers_dataset.csv'
topic_name = 'raw_customers'

print(f"Đang đọc dữ liệu từ {csv_file_path}...")
try:
    df = pd.read_csv(csv_file_path)
    print(f"Đã đọc thành công {len(df)} dòng dữ liệu!")
    
    # 3. Vòng lặp bắn dữ liệu (Giả lập Streaming)
    print(f"Bắt đầu đẩy luồng sự kiện vào topic: {topic_name}")
    count = 0
    
    for index, row in df.iterrows():
        # Chuyển mỗi dòng thành 1 sự kiện dạng JSON (Dictionary)
        record = row.dropna().to_dict() 
        
        # Bắn sự kiện vào Kafka
        producer.send(topic_name, value=record)
        count += 1
        
        # In tiến độ ra màn hình & Xả bộ đệm mỗi 1000 dòng
        if count % 1000 == 0:
            print(f"Đã gửi {count} bản ghi...")
            producer.flush()
            
        # Nghỉ 0.01 giây giữa mỗi lần bắn để giả lập hệ thống Real-time
        time.sleep(0.01)

    producer.flush()
    print("🎉 Hoàn tất đẩy toàn bộ dữ liệu!")
    
except FileNotFoundError:
    print(f"❌ LỖI: Không tìm thấy file tại {csv_file_path}.")
    print("Vui lòng kiểm tra lại xem file olist_customers_dataset.csv đã nằm trong thư mục data_source chưa!")


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Đang đọc dữ liệu từ /home/iceberg/data/olist_customers_dataset.csv...
Đã đọc thành công 99441 dòng dữ liệu!
Bắt đầu đẩy luồng sự kiện vào topic: raw_customers
Đã gửi 1000 bản ghi...
Đã gửi 2000 bản ghi...
Đã gửi 3000 bản ghi...
Đã gửi 4000 bản ghi...
Đã gửi 5000 bản ghi...
Đã gửi 6000 bản ghi...
Đã gửi 7000 bản ghi...
Đã gửi 8000 bản ghi...
Đã gửi 9000 bản ghi...
Đã gửi 10000 bản ghi...
Đã gửi 11000 bản ghi...
Đã gửi 12000 bản ghi...
Đã gửi 13000 bản ghi...
Đã gửi 14000 bản ghi...
Đã gửi 15000 bản ghi...
Đã gửi 16000 bản ghi...
Đã gửi 17000 bản ghi...
Đã gửi 18000 bản ghi...
Đã gửi 19000 bản ghi...
Đã gửi 20000 bản ghi...
Đã gửi 21000 bản ghi...
Đã gửi 22000 bản ghi...
Đã gửi 23000 bản ghi...
Đã gửi 24000 bản ghi...
Đã gửi 25000 bản ghi...
Đã gửi 26000 bản ghi...
Đã gửi 27000 bản ghi...
Đã gửi 28000 bản ghi...
Đã gửi 29000 bản ghi...
Đã gửi 30000 bản ghi...
Đã gửi 31000 bản 